In [ ]:
#Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.utils import shuffle
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

SEED = 13
N_SPLITS = 5

DATA_DIR = Path("/content/drive/My Drive/Colab Notebooks/TFM/DataSet")
INPUT_PATH = DATA_DIR / "05_text_speech_eeg.csv"
PARTITIONS_PATH = DATA_DIR / "data_partitions_paper_ready.csv"


In [ ]:
def load_partitions():
    """Carga directamente las particiones del paper."""
    partitions = pd.read_csv(PARTITIONS_PATH)
    return partitions[["subject_id", "avatar", "outer_fold"]].copy()


def get_metrics(y_true, y_pred, y_prob):
    return {
        "WAcc": accuracy_score(y_true, y_pred),
        "UAcc": balanced_accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


def subject_level_predictions(pred_conv):
    pred_subject = (
        pred_conv
        .groupby(["subject_id", "label", "outer_fold"], as_index=False)["prob_1"]
        .mean()
    )
    pred_subject["pred"] = (pred_subject["prob_1"] >= 0.5).astype(int)
    return pred_subject


def summarize_mean_std(df, cols):
    return pd.concat([
        df[cols].mean().round(3).rename("mean"),
        df[cols].std().round(3).rename("std"),
    ], axis=1)


In [ ]:
data = pd.read_csv(INPUT_PATH)
partitions = load_partitions()

text_cols = sorted([c for c in data.columns if c.startswith("text_")], key=lambda c: int(c.split("_", 1)[1]))
speech_cols = sorted([c for c in data.columns if c.startswith("speech_")], key=lambda c: int(c.split("_", 1)[1]))
meta_cols = {"subject_id", "avatar", "label"}
eeg_cols = [c for c in data.columns if c not in meta_cols and c not in text_cols and c not in speech_cols]
feature_cols = text_cols + speech_cols + eeg_cols

required = {"subject_id", "avatar", "label"}
if not required.issubset(data.columns):
    raise ValueError(f"Faltan columnas obligatorias: {required - set(data.columns)}")


partitions = shuffle(partitions, random_state=SEED).reset_index(drop=True)
df = data.merge(partitions, on=["subject_id", "avatar"], how="inner")

n_before = len(df)
df = df.dropna(subset=feature_cols).copy()
n_removed = n_before - len(df)

print("Filas iniciales con partición:", n_before)
print("Filas eliminadas por no tener alguna modalidad:", n_removed)
print("Filas trimodales finales:", len(df))
print("Sujetos finales:", df["subject_id"].nunique())
print("Variables text:", len(text_cols))
print("Variables speech:", len(speech_cols))
print("Variables EEG:", len(eeg_cols))

print("\nSujetos por outer fold después del filtro:")
display(df.groupby("outer_fold")["subject_id"].nunique().to_frame("n_subjects"))

print("\nDistribución de clases por outer fold:")
display(pd.crosstab(df.drop_duplicates("subject_id")["outer_fold"], df.drop_duplicates("subject_id")["label"]))

print("\nConversaciones disponibles por narrativa:")
display(df["avatar"].value_counts().rename_axis("avatar").to_frame("n_rows"))


Filas iniciales con partición: 600
Filas eliminadas por no tener alguna modalidad: 42
Filas trimodales finales: 558
Sujetos finales: 94
Variables text: 768
Variables speech: 1024
Variables EEG: 27

Sujetos por outer fold después del filtro:


,n_subjects
outer_fold,
1,20
2,18
3,18
4,19
5,19



Distribución de clases por outer fold:


label,0,1
outer_fold,,
1,11,9
2,10,8
3,11,7
4,12,7
5,11,8



Conversaciones disponibles por narrativa:


,n_rows
avatar,
Sad,94
Neutral1,94
Happy,94
Angry,92
Relax,92
Neutral2,92


In [5]:
param_grid = {
    "max_depth": list(range(3, 12)),
    "n_estimators": [25, 50, 100, 200],
}

scoring = {
    "WAcc": "accuracy",
    "UAcc": "balanced_accuracy",
    "auc": "roc_auc",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
}

metric_cols = ["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]


In [ ]:
OUT_DIR = DATA_DIR / "05_results_text_speech_eeg_emotion_wise"
OUT_DIR.mkdir(parents=True, exist_ok=True)

emotion_metrics_rows = []
subject_metrics_rows = []
best_params_rows = []
all_emotion_predictions = []

emotions = sorted(df["avatar"].unique())

for fold in sorted(df["outer_fold"].unique()):
    print(f"\n===== OUTER FOLD {fold} =====")
    fold_predictions = []

    for emotion in emotions:
        emo_df = df[df["avatar"] == emotion].copy()
        dev = emo_df[emo_df["outer_fold"] != fold].reset_index(drop=True)
        test = emo_df[emo_df["outer_fold"] == fold].reset_index(drop=True)

        if len(test) == 0 or dev["label"].nunique() < 2 or test["label"].nunique() < 2:
            print(f"{emotion}: omitido en fold {fold} por falta de datos/clases")
            continue

        X_dev = dev[feature_cols].to_numpy(dtype=np.float32)
        y_dev = dev["label"].to_numpy(dtype=int)
        groups_dev = dev["subject_id"].to_numpy()

        X_test = test[feature_cols].to_numpy(dtype=np.float32)
        y_test = test["label"].to_numpy(dtype=int)

        inner_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

        grid = GridSearchCV(
            estimator=XGBClassifier(),
            param_grid=param_grid,
            scoring=scoring,
            refit="UAcc",
            cv=inner_cv,
            n_jobs=-1,
            verbose=0,
        )
        grid.fit(X_dev, y_dev, groups=groups_dev)

        best_params = grid.best_params_
        best_idx = grid.best_index_

        model = XGBClassifier(**best_params)
        model.fit(X_dev, y_dev)

        prob_1 = model.predict_proba(X_test)[:, 1]
        pred = (prob_1 >= 0.5).astype(int)

        pred_emo = test[["subject_id", "avatar", "label", "outer_fold"]].copy()
        pred_emo["prob_1"] = prob_1
        pred_emo["pred"] = pred
        fold_predictions.append(pred_emo)
        all_emotion_predictions.append(pred_emo)

        emo_metrics = get_metrics(y_test, pred, prob_1)
        emo_metrics["outer_fold"] = fold
        emo_metrics["avatar"] = emotion
        emo_metrics["cv_f1"] = grid.cv_results_["mean_test_f1"][best_idx]
        emotion_metrics_rows.append(emo_metrics)

        best_params_rows.append({
            "outer_fold": fold,
            "avatar": emotion,
            "best_max_depth": best_params["max_depth"],
            "best_n_estimators": best_params["n_estimators"],
            "best_inner_UAcc": grid.cv_results_["mean_test_UAcc"][best_idx],
            "best_inner_f1": grid.cv_results_["mean_test_f1"][best_idx],
        })

        print(f"{emotion}: CV F1={grid.cv_results_['mean_test_f1'][best_idx]:.3f} | Test F1={emo_metrics['f1']:.3f}")

    # Agregacion por sujeto usando las probabilidades disponibles de las narrativas de ese fold
    fold_pred = pd.concat(fold_predictions, ignore_index=True)
    pred_subject = subject_level_predictions(fold_pred)
    subject_metrics = get_metrics(pred_subject["label"], pred_subject["pred"], pred_subject["prob_1"])
    subject_metrics["outer_fold"] = fold
    subject_metrics["n_subjects"] = pred_subject["subject_id"].nunique()
    subject_metrics_rows.append(subject_metrics)

    print("Subject-level Test F1 agregado:", round(subject_metrics["f1"], 3))

emotion_metrics_df = pd.DataFrame(emotion_metrics_rows)
subject_metrics_df = pd.DataFrame(subject_metrics_rows)
best_params_df = pd.DataFrame(best_params_rows)
emotion_predictions_df = pd.concat(all_emotion_predictions, ignore_index=True)
subject_predictions_global = subject_level_predictions(emotion_predictions_df)
global_subject_metrics = get_metrics(
    subject_predictions_global["label"],
    subject_predictions_global["pred"],
    subject_predictions_global["prob_1"],
)

subject_summary = pd.DataFrame({
    "metric": ["Subject-level Test F1"],
    "mean": [subject_metrics_df["f1"].mean()],
    "std": [subject_metrics_df["f1"].std()],
}).round(3)

emotion_summary = (
    emotion_metrics_df
    .groupby("avatar")[["cv_f1", "f1", "UAcc", "auc"]]
    .agg(["mean", "std"])
    .round(3)
)

emotion_metrics_df.to_csv(OUT_DIR / "emotion_level_outer_metrics.csv", index=False)
subject_metrics_df.to_csv(OUT_DIR / "subject_level_outer_metrics.csv", index=False)
best_params_df.to_csv(OUT_DIR / "best_params_by_outer_fold_and_emotion.csv", index=False)
emotion_predictions_df.to_csv(OUT_DIR / "emotion_predictions.csv", index=False)
subject_predictions_global.to_csv(OUT_DIR / "subject_predictions_global.csv", index=False)
subject_summary.to_csv(OUT_DIR / "main_subject_level_summary.csv", index=False)

print("\nResumen por narrativa")
display(emotion_summary)

print("\nResultado principal agregado por sujeto")
display(subject_summary)

print("\nMétricas subject-level globales")
display(pd.Series(global_subject_metrics).round(3).to_frame("global"))

print("\nArchivos guardados en:", OUT_DIR)



===== OUTER FOLD 1 =====
Angry: CV F1=0.497 | Test F1=0.615
Happy: CV F1=0.359 | Test F1=0.526
Neutral1: CV F1=0.531 | Test F1=0.700
Neutral2: CV F1=0.437 | Test F1=0.625
Relax: CV F1=0.591 | Test F1=0.588
Sad: CV F1=0.382 | Test F1=0.625
Subject-level Test F1 agregado: 0.75

===== OUTER FOLD 2 =====
Angry: CV F1=0.484 | Test F1=0.462
Happy: CV F1=0.443 | Test F1=0.267
Neutral1: CV F1=0.461 | Test F1=0.471
Neutral2: CV F1=0.392 | Test F1=0.462
Relax: CV F1=0.669 | Test F1=0.308
Sad: CV F1=0.512 | Test F1=0.400
Subject-level Test F1 agregado: 0.533

===== OUTER FOLD 3 =====
Angry: CV F1=0.331 | Test F1=0.769
Happy: CV F1=0.546 | Test F1=0.571
Neutral1: CV F1=0.531 | Test F1=0.429
Neutral2: CV F1=0.377 | Test F1=0.000
Relax: CV F1=0.497 | Test F1=0.500
Sad: CV F1=0.434 | Test F1=0.444
Subject-level Test F1 agregado: 0.6

===== OUTER FOLD 4 =====
Angry: CV F1=0.520 | Test F1=0.364
Happy: CV F1=0.401 | Test F1=0.429
Neutral1: CV F1=0.460 | Test F1=0.769
Neutral2: CV F1=0.369 | Test F1=0.5

cv_f1            f1          UAcc           auc       
           mean    std   mean    std   mean    std   mean    std
avatar                                                          
Angry     0.448  0.078  0.592  0.178  0.687  0.119  0.672  0.159
Happy     0.411  0.091  0.451  0.117  0.544  0.103  0.593  0.085
Neutral1  0.495  0.035  0.507  0.239  0.596  0.160  0.651  0.136
Neutral2  0.401  0.031  0.417  0.241  0.580  0.088  0.570  0.139
Relax     0.578  0.066  0.471  0.145  0.585  0.097  0.630  0.091
Sad       0.396  0.088  0.535  0.127  0.629  0.106  0.633  0.102


Resultado principal agregado por sujeto


,metric,mean,std
0,Subject-level Test F1,0.583,0.107



Métricas subject-level globales


,global
WAcc,0.702
UAcc,0.675
auc,0.710
f1,0.588
precision,0.690
recall,0.513
kappa,0.363



Archivos guardados en: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/05_results_text_speech_eeg_emotion_wise
